# ACC102 Track 4 - Stock Data Analysis Notebook

**Pipeline**: Data Collection -> Cleaning -> Transformation -> Analysis -> Visualisation -> Export

**Stocks**: AAPL, MSFT, TSLA, NVDA, GOOGL, AMZN, META, JPM, KO, WMT (10 stocks)
**Period**: 2022-2024 | **Source**: Yahoo Finance | **Rows**: 7,530

In [1]:
# ============================================================
# STEP 0: Import Libraries
# ============================================================
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

## Step 1: Data Collection

In [2]:
# ============================================================
# STEP 1: Collect Data from Yahoo Finance
# Download OHLCV + Dividends for 10 stocks (2022-2024)
# ============================================================

STOCKS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL', 'AMZN', 'META', 'JPM', 'KO', 'WMT']
start_date = '2022-01-01'
end_date = '2024-12-31'

all_data = []
for sym in STOCKS:
    ticker = yf.Ticker(sym)
    df = ticker.history(start=start_date, end=end_date, progress=False)
    df.reset_index(inplace=True)
    df['Symbol'] = sym
    all_data.append(df)
    print(f"{sym}: {len(df)} rows downloaded")

combined = pd.concat(all_data, ignore_index=True)
combined = combined[['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends']].copy()
combined['Date'] = pd.to_datetime(combined['Date']).dt.strftime('%Y-%m-%d')
print(f"\nTotal combined: {len(combined)} rows for {combined['Symbol'].nunique()} stocks")

## Step 2: Data Cleaning & Transformation

In [3]:
# ============================================================
# STEP 2: Clean and Transform Data
# ============================================================

# 2.1 Ensure price integrity: High >= Close >= Low
combined['High'] = combined[['High', 'Close', 'Open']].max(axis=1)
combined['Low'] = combined[['Low', 'Close', 'Open']].min(axis=1)

# 2.2 Check for nulls and data types
print("Data Quality Report:")
print(f"  Stocks: {sorted(combined['Symbol'].unique().tolist())}")
print(f"  Date range: {combined['Date'].min()} to {combined['Date'].max()}")
print(f"  Total rows: {len(combined)}")
print(f"  Null values: {combined.isnull().sum().sum()}")
print(f"  Rows per stock: {len(combined) // combined['Symbol'].nunique()}")

# 2.3 Summary statistics
print("\nClose Price Summary by Stock:")
combined.groupby('Symbol')['Close'].agg(['count','mean','min','max']).round(2)

## Step 3a: Daily Indicators Calculation

In [4]:
# ============================================================
# STEP 3a: Calculate Daily Technical Indicators
# MA(20), MA(60), RSI, Drawdown, Volatility, MA deviation
# ============================================================

def calc_daily_indicators(df):
    df = df.sort_values('Date').copy()
    df['Daily_Return_%'] = df['Close'].pct_change() * 100
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA60'] = df['Close'].rolling(60).mean()
    df['vs_MA20_%'] = ((df['Close'] - df['MA20']) / df['MA20'] * 100).round(2)
    df['vs_MA60_%'] = ((df['Close'] - df['MA60']) / df['MA60'] * 100).round(2)
    df['Volatility_20d'] = (df['Daily_Return_%'].rolling(20).std() * np.sqrt(252)).round(2)
    cummax = df['Close'].cummax()
    df['Drawdown_%'] = ((df['Close'] - cummax) / cummax * 100).round(2)
    # RSI
    delta = df['Close'].diff()
    gain, loss = delta.clip(lower=0), -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / avg_loss
    df['RSI'] = (100 - (100 / (1 + rs))).round(2)
    return df

daily_parts = []
for s in STOCKS:
    sdf = combined[combined['Symbol'] == s].copy()
    sdf = calc_daily_indicators(sdf)
    daily_parts.append(sdf)

daily_full = pd.concat(daily_parts, ignore_index=True)
print(f"Daily indicators calculated: {len(daily_full)} rows")
print(f"Columns: {list(daily_full.columns)}")
daily_full[['Date','Symbol','Close','Daily_Return_%','RSI','vs_MA20_%','Drawdown_%']].head()

## Step 3b: Monthly Returns

In [5]:
# ============================================================
# STEP 3b: Calculate Monthly Returns
# ============================================================

monthly_parts = []
for s in STOCKS:
    sdf = combined[combined['Symbol'] == s].copy()
    sdf['Date_dt'] = pd.to_datetime(sdf['Date'])
    sdf.set_index('Date_dt', inplace=True)
    monthly = sdf['Close'].resample('ME').last().to_frame()
    monthly['Monthly_Return_%'] = (monthly['Close'].pct_change() * 100).round(2)
    monthly['Symbol'] = s
    monthly.reset_index(inplace=True)
    monthly['Date'] = monthly['Date_dt'].dt.strftime('%Y-%m')
    monthly_parts.append(monthly[['Date','Symbol','Close','Monthly_Return_%']].dropna())

monthly_all = pd.concat(monthly_parts, ignore_index=True)
print(f"Monthly returns: {len(monthly_all)} rows")
monthly_all.head(10)

## Step 3c: Yearly Summary

In [6]:
# ============================================================
# STEP 3c: Calculate Yearly Summary (Return / Risk / Dividends)
# ============================================================

yearly_rows = []
for s in STOCKS:
    sdf = combined[combined['Symbol'] == s].copy()
    sdf['Year'] = pd.to_datetime(sdf['Date']).dt.year
    for yr in [2022, 2023, 2024]:
        ydf = sdf[sdf['Year'] == yr]
        if len(ydf) == 0: continue
        sp, ep = ydf['Close'].iloc[0], ydf['Close'].iloc[-1]
        dr = ydf['Close'].pct_change().dropna()
        yearly_rows.append({
            'Year': yr, 'Symbol': s, 'Start_Price': round(sp,2), 'End_Price': round(ep,2),
            'Annual_Return_%': round(((ep-sp)/sp)*100, 2),
            'Volatility_%': round(dr.std()*np.sqrt(252)*100, 2),
            'Max_Drawdown_%': round(((ydf['Close']-ydf['Close'].cummax())/ydf['Close'].cummax()).min()*100, 2),
            'Total_Dividend': round(ydf['Dividends'].sum(), 4),
            'Dividend_Count': int((ydf['Dividends']>0).sum()),
            'Trading_Days': len(ydf)
        })

yearly_df = pd.DataFrame(yearly_rows)
print(f"Yearly summary: {len(yearly_df)} rows")
yearly_df.head(12)

## Step 3d: Full-Period Summary (Returns / Momentum / Dividends)

In [7]:
# ============================================================
# STEP 3d: Full-Period Summary Analysis
# ============================================================

analysis_results, momentum_data, dividend_data = [], [], []

for s in STOCKS:
    sdf = combined[combined['Symbol'] == s].sort_values('Date')
    closes = sdf['Close'].values
    sp, ep = float(closes[0]), float(closes[-1])
    tr = ((ep-sp)/sp)*100
    years = (pd.to_datetime(sdf['Date'].iloc[-1]) - pd.to_datetime(sdf['Date'].iloc[0])).days/365.25
    ar = (((ep/sp)**(1/years))-1)*100
    analysis_results.append({'Stock':s,'Start_Price':round(sp,2),'End_Price':round(ep,2),
        'Total_Return_%':round(tr,2),'Annual_Return_%':round(ar,2),'Trading_Days':len(closes)})
    
    dr = np.diff(closes)/closes[:-1]
    vol = float(np.std(dr)*np.sqrt(252)*100)
    cummax = np.maximum.accumulate(closes)
    mdd = float(np.min((closes-cummax)/cummax)*100)
    ma20 = float(np.mean(closes[-20:])); ma60 = float(np.mean(closes[-60:]))
    def pr(d): return ((closes[-1]-closes[-(d+1)])/closes[-(d+1)])*100 if len(closes)>d else 0
    momentum_data.append({'Stock':s,'Volatility_%':round(vol,2),'Max_Drawdown_%':round(mdd,2),
        'vs_MA20_%':round(((closes[-1]-ma20)/ma20)*100,2),'vs_MA60_%':round(((closes[-1]-ma60)/ma60)*100,2),
        '1M_%':round(pr(21),2),'3M_%':round(pr(63),2),'6M_%':round(pr(126),2),'1Y_%':round(pr(252),2)})
    
    td = float(sdf['Dividends'].sum())
    dc = int((sdf['Dividends']>0).sum())
    rd = 0.0
    for v in reversed(sdf['Dividends'].values):
        if v > 0: rd = float(v); break
    dividend_data.append({'Stock':s,'Annual_Dividend':round(td,4),
        'Dividend_Yield_%':round((td/ep)*100,2),'Payout_Count':dc,'Latest_Dividend':round(rd,4)})

result_df = pd.DataFrame(analysis_results)
momentum_df = pd.DataFrame(momentum_data)
dividend_df = pd.DataFrame(dividend_data)

print("=== RETURN ANALYSIS ===")
print(result_df.to_string(index=False))
print("\n=== MOMENTUM INDICATORS ===")
print(momentum_df.to_string(index=False))
print("\n=== DIVIDEND DATA ===")
print(dividend_df.to_string(index=False))

## Step 4: Visualisation (6-Panel Dashboard)

In [8]:
# ============================================================
# STEP 4: 6-Panel Visualisation Dashboard
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# 1. Total Return Bar Chart
colors = ['#27ae60' if x > 0 else '#e74c3c' for x in result_df['Total_Return_%']]
axes[0,0].bar(result_df['Stock'], result_df['Total_Return_%'], color=colors)
axes[0,0].set_title('Total Return Comparison (%)', fontweight='bold', fontsize=12)
axes[0,0].axhline(y=0, color='black', linewidth=0.5)
axes[0,0].grid(True, alpha=0.3, axis='y')
axes[0,0].tick_params(axis='x', rotation=45)

# 2. Normalised Price Trend
for s in STOCKS:
    d = combined[combined['Symbol'] == s].sort_values('Date')
    norm = d['Close'] / d['Close'].iloc[0] * 100
    axes[0,1].plot(range(len(norm)), norm, label=s, linewidth=1.5, alpha=0.8)
axes[0,1].set_title('Price Trend (Normalised = 100)', fontweight='bold', fontsize=12)
axes[0,1].legend(loc='upper left', fontsize=7, ncol=2)
axes[0,1].grid(True, alpha=0.3)

# 3. Volatility Comparison
axes[0,2].bar(momentum_df['Stock'], momentum_df['Volatility_%'], color='#3498db', alpha=0.8)
axes[0,2].set_title('Volatility (%)', fontweight='bold', fontsize=12)
axes[0,2].grid(True, alpha=0.3, axis='y')
axes[0,2].tick_params(axis='x', rotation=45)

# 4. Max Drawdown
axes[1,0].bar(momentum_df['Stock'], momentum_df['Max_Drawdown_%'], color='#e74c3c', alpha=0.8)
axes[1,0].set_title('Max Drawdown (%)', fontweight='bold', fontsize=12)
axes[1,0].grid(True, alpha=0.3, axis='y')
axes[1,0].tick_params(axis='x', rotation=45)

# 5. Dividend Yield
colors_div = ['#27ae60' if x > 0 else '#95a5a6' for x in dividend_df['Dividend_Yield_%']]
axes[1,1].bar(dividend_df['Stock'], dividend_df['Dividend_Yield_%'], color=colors_div)
axes[1,1].set_title('Dividend Yield (%)', fontweight='bold', fontsize=12)
axes[1,1].grid(True, alpha=0.3, axis='y')
axes[1,1].tick_params(axis='x', rotation=45)

# 6. MA Deviation
x = np.arange(len(momentum_df))
w = 0.35
axes[1,2].bar(x - w/2, momentum_df['vs_MA20_%'], w, label='vs MA20 %')
axes[1,2].bar(x + w/2, momentum_df['vs_MA60_%'], w, label='vs MA60 %')
axes[1,2].set_title('Moving Average Deviation (%)', fontweight='bold', fontsize=12)
axes[1,2].set_xticks(x)
axes[1,2].set_xticklabels(momentum_df['Stock'], rotation=45)
axes[1,2].axhline(y=0, color='black', linewidth=0.5)
axes[1,2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('figures/analysis_dashboard_10stocks.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to figures/analysis_dashboard_10stocks.png')

## Step 5: Export to Excel Database

In [9]:
# ============================================================
# STEP 5: Export Complete Database to Excel (7 tables)
# ============================================================

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment

output_file = 'data/stock_database.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    combined.to_excel(writer, sheet_name='stock_data', index=False)
    daily_full[['Date','Symbol','Close','Daily_Return_%','RSI','vs_MA20_%','vs_MA60_%',
                'Drawdown_%','Volatility_20d','MA20','MA60']].to_excel(
        writer, sheet_name='daily_indicators', index=False)
    monthly_all.to_excel(writer, sheet_name='monthly_returns', index=False)
    yearly_df.to_excel(writer, sheet_name='yearly_summary', index=False)
    result_df.to_excel(writer, sheet_name='analysis_results', index=False)
    momentum_df.to_excel(writer, sheet_name='momentum_indicators', index=False)
    dividend_df.to_excel(writer, sheet_name='dividend_data', index=False)

print(f'Database exported: {output_file}')
print(f'  stock_data: {len(combined)} rows')
print(f'  daily_indicators: {len(daily_full)} rows')
print(f'  monthly_returns: {len(monthly_all)} rows')
print(f'  yearly_summary: {len(yearly_df)} rows')
print(f'  analysis_results: {len(result_df)} rows')
print(f'  momentum_indicators: {len(momentum_df)} rows')
print(f'  dividend_data: {len(dividend_df)} rows')
print(f'  TOTAL: {len(combined) + len(daily_full) + len(monthly_all) + len(yearly_df) + len(result_df) + len(momentum_df) + len(dividend_df)} rows')

---

## Summary

This notebook demonstrates the complete data pipeline for ACC102 Track 4:

1. **Collection**: Downloaded 7,530 rows of OHLCV + dividend data for 10 stocks from Yahoo Finance
2. **Cleaning**: Validated price integrity (High >= Close >= Low), checked nulls, standardised formats
3. **Transformation**: Calculated daily indicators (MA20/MA60, RSI, drawdown, volatility, MA deviation)
4. **Analysis**: Computed monthly returns, yearly summaries, and full-period metrics (returns, risk, momentum, dividends)
5. **Visualisation**: Generated a 6-panel dashboard comparing all 10 stocks across multiple dimensions
6. **Export**: Saved structured database to Excel with 7 analysis tables (15,470 total rows)

**Key findings**: NVDA delivered exceptional returns (+490% over 3 years) driven by AI demand, while KO provided the most stable performance (lowest volatility at 18.4%). TSLA exhibited extreme volatility (61%) with a -73% max drawdown, highlighting the risk-reward trade-off in tech investing.